In [50]:
import openai
import os
from superlinked import framework as sl
import json
from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue
from langsmith import Client


### Download data from qdrant

In [ ]:
# Superlinked's `QdrantVectorDatabase` is only for Superlinked executors (index/query).
# To list or export points, use the official Qdrant client and `scroll`.
qdrant = QdrantClient(
    url="http://localhost:6333",
    api_key="",  # empty for local Qdrant; use your cloud API key for Qdrant Cloud
)

/var/folders/rt/bmzmdtl92k7bz0smp4yx2f7r0000gn/T/ipykernel_72203/2146870198.py:5: UserWarning: Api key is used with an insecure connection.
  qdrant = QdrantClient(


In [8]:
COLLECTION = "yelp-businesses-collection-00"

# One page (up to `limit` points). For the full collection, loop while `next_offset` is not None.
all_data= qdrant.scroll(
    collection_name=COLLECTION,
    limit=100,
    offset=None,
    with_vectors=False,
    with_payload=True,
)

In [9]:
all_data

([Record(id='000750dc-a308-573b-9183-c64e2102a99b', payload={'__schema_field__Business_name': 'Roll DogZ', '__schema_field__Business_address': '22 N Darlington St', '__schema_field__Business_city': 'West Chester', '__schema_field__Business_state': 'PA', '__schema_field__Business_postal_code': '19380', '__schema_field__Business_latitude': 39.9589982, '__schema_field__Business_longitude': -75.6076348, '__schema_field__Business_stars': 2.5, '__schema_field__Business_review_count': 11, '__schema_field__Business_is_open': False, '__schema_field__Business_is_open_i': 0, '__schema_field__Business_attributes': '{"Alcohol": "u\'none\'", "Ambience": "{\'romantic\': False, \'intimate\': False, \'touristy\': False, \'hipster\': False, \'divey\': False, \'classy\': False, \'trendy\': False, \'upscale\': False, \'casual\': False}", "BikeParking": "False", "BusinessAcceptsCreditCards": "True", "BusinessParking": "{\'garage\': False, \'street\': True, \'validated\': False, \'lot\': False, \'valet\': F

In [12]:
all_data[0][0].payload

{'__schema_field__Business_name': 'Roll DogZ',
 '__schema_field__Business_address': '22 N Darlington St',
 '__schema_field__Business_city': 'West Chester',
 '__schema_field__Business_state': 'PA',
 '__schema_field__Business_postal_code': '19380',
 '__schema_field__Business_latitude': 39.9589982,
 '__schema_field__Business_longitude': -75.6076348,
 '__schema_field__Business_stars': 2.5,
 '__schema_field__Business_review_count': 11,
 '__schema_field__Business_is_open': False,
 '__schema_field__Business_is_open_i': 0,
 '__schema_field__Business_attributes': '{"Alcohol": "u\'none\'", "Ambience": "{\'romantic\': False, \'intimate\': False, \'touristy\': False, \'hipster\': False, \'divey\': False, \'classy\': False, \'trendy\': False, \'upscale\': False, \'casual\': False}", "BikeParking": "False", "BusinessAcceptsCreditCards": "True", "BusinessParking": "{\'garage\': False, \'street\': True, \'validated\': False, \'lot\': False, \'valet\': False}", "Caters": "True", "GoodForKids": "True", 

### Render a prompt to generate synthetic Eval reference dataset

In [19]:
all_context =[{"id":data.payload["__object_id__"], "name":data.payload["__schema_field__Business_name"], "city":data.payload["__schema_field__Business_city"]} for data in all_data[0]]

In [25]:
all_context = [
    {
        "id": data.payload.get("__object_id__"),
        **{
            k.replace("__schema_field__Business_", "", 1): v
            for k, v in data.payload.items()
            if k.startswith("__schema_field__Business_")
        },
    }
    for data in all_data[0]   # keep this if you did: all_data = qdrant.scroll(...)
]

In [26]:
all_context

[{'id': '2mLIChoI05JUG_vSDygwag',
  'name': 'Roll DogZ',
  'address': '22 N Darlington St',
  'city': 'West Chester',
  'state': 'PA',
  'postal_code': '19380',
  'latitude': 39.9589982,
  'longitude': -75.6076348,
  'stars': 2.5,
  'review_count': 11,
  'is_open': False,
  'is_open_i': 0,
  'attributes': '{"Alcohol": "u\'none\'", "Ambience": "{\'romantic\': False, \'intimate\': False, \'touristy\': False, \'hipster\': False, \'divey\': False, \'classy\': False, \'trendy\': False, \'upscale\': False, \'casual\': False}", "BikeParking": "False", "BusinessAcceptsCreditCards": "True", "BusinessParking": "{\'garage\': False, \'street\': True, \'validated\': False, \'lot\': False, \'valet\': False}", "Caters": "True", "GoodForKids": "True", "HasTV": "True", "NoiseLevel": "u\'average\'", "OutdoorSeating": "False", "RestaurantsAttire": "u\'casual\'", "RestaurantsDelivery": "True", "RestaurantsGoodForGroups": "True", "RestaurantsPriceRange2": "1", "RestaurantsReservations": "False", "Restauran

In [37]:
output_schema = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "Suggested question.",
            },
            "chunk_ids": {
                "type": "array",
                "items": {
                    "type": "string",
                    "description": "ID of the chunk that could be used to answer the question.",
                },
            },
            "answer_example": {
                "type": "string",
                "description": "Suggested answer grounded in the context.",
            },
            "reasoning": {
                "type": "string",
                "description": "Reasoning why the question could be answered with the chunks.",
            },
        },
    },
}

SYSTEM_PROMPT = f"""
I am building a RAG application. I have a collection of 100 chunks of business data.
The RAG application will act as a yelp shopping assistant that can answer questions about specific restaurants available in a city.
I will provide all of the available restaurants to you with IDs of each chunk.
I want you to come up with 60 questions to which the answers could be grounded in the chunk context.
The questions should imitate a potential real user of this RAG system.
As an output I need you to provide me the list of questions and the IDs of the chunks that could be used to answer them.
Also, provide an example answer to the question given the context of the chunks.
Also, provide the reason why you chose the chunks to answer the questions.
The questions need to reflect typical questions that a user would ask to find a restaurant in a city.
Construct 40 questions that could use multipple chunks in the answer.
Construct 10 questions that could use single chunk in the answer.
Construct 10 questions that can't be answered with the available chunks.

<OUTPUT JSON SCHEMA>
{json.dumps(output_schema, indent=2)}
</OUTPUT JSON SCHEMA>

I need to be able to parse the json output.
"""

USER_PROMPT = f"""
Here is the list of chunks, each list element is a dictionary with restaurants ids and restaurant data:
{all_context}
"""

In [38]:
print(SYSTEM_PROMPT)


I am building a RAG application. I have a collection of 100 chunks of business data.
The RAG application will act as a yelp shopping assistant that can answer questions about specific restaurants available in a city.
I will provide all of the available restaurants to you with IDs of each chunk.
I want you to come up with 60 questions to which the answers could be grounded in the chunk context.
The questions should imitate a potential real user of this RAG system.
As an output I need you to provide me the list of questions and the IDs of the chunks that could be used to answer them.
Also, provide an example answer to the question given the context of the chunks.
Also, provide the reason why you chose the chunks to answer the questions.
The questions need to reflect typical questions that a user would ask to find a restaurant in a city.
Construct 40 questions that could use multipple chunks in the answer.
Construct 10 questions that could use single chunk in the answer.
Construct 10 que

In [39]:
print(USER_PROMPT)


Here is the list of chunks, each list element is a dictionary with restaurants ids and restaurant data:
[{'id': '2mLIChoI05JUG_vSDygwag', 'name': 'Roll DogZ', 'address': '22 N Darlington St', 'city': 'West Chester', 'state': 'PA', 'postal_code': '19380', 'latitude': 39.9589982, 'longitude': -75.6076348, 'stars': 2.5, 'review_count': 11, 'is_open': False, 'is_open_i': 0, 'attributes': '{"Alcohol": "u\'none\'", "Ambience": "{\'romantic\': False, \'intimate\': False, \'touristy\': False, \'hipster\': False, \'divey\': False, \'classy\': False, \'trendy\': False, \'upscale\': False, \'casual\': False}", "BikeParking": "False", "BusinessAcceptsCreditCards": "True", "BusinessParking": "{\'garage\': False, \'street\': True, \'validated\': False, \'lot\': False, \'valet\': False}", "Caters": "True", "GoodForKids": "True", "HasTV": "True", "NoiseLevel": "u\'average\'", "OutdoorSeating": "False", "RestaurantsAttire": "u\'casual\'", "RestaurantsDelivery": "True", "RestaurantsGoodForGroups": "Tru

In [40]:
response = openai.chat.completions.create(
    model="gpt-5-mini",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT}
    ],
    reasoning_effort="minimal"
)

print(response.choices[0].message.content)

[
  {
    "question": "Which restaurants in Philadelphia offer outdoor seating and accept credit cards?",
    "chunk_ids": [
      "DhZVpgeBKxoLd91MiwTxpg",
      "ZNB73mij2iG0wfbXBazQdw",
      "oXr3EhnQCqA8SNWIZ3H4Fg",
      "7M4acncueqDznH9AwgPjeg"
    ],
    "answer_example": "La Fontana Della Citta (1701 Spruce St) offers outdoor seating and accepts credit cards. Green Engine Coffee (16 Haverford Station Rd) has outdoor seating and accepts credit cards. Little Spoon Cafe (1500 South St) also has outdoor seating and accepts credit cards. Maggie's Waterfront Cafe (9242 N Delaware Ave) offers outdoor seating and accepts credit cards.",
    "reasoning": "These chunks are Philadelphia restaurants whose attributes list OutdoorSeating = True and BusinessAcceptsCreditCards = True (or bike_parking/accepts_credit_cards flags)."
  },
  {
    "question": "Find family-friendly places with good for kids in Tampa.",
    "chunk_ids": [
      "ZXt0zo8liURaAEKD47zDRQ",
      "c7ZxVUJVjEOe9hLhvKGnDA

In [41]:
import json

json_output = response.choices[0].message.content
json_output = json.loads(json_output)

In [42]:
json_output

[{'question': 'Which restaurants in Philadelphia offer outdoor seating and accept credit cards?',
  'chunk_ids': ['DhZVpgeBKxoLd91MiwTxpg',
   'ZNB73mij2iG0wfbXBazQdw',
   'oXr3EhnQCqA8SNWIZ3H4Fg',
   '7M4acncueqDznH9AwgPjeg'],
  'answer_example': "La Fontana Della Citta (1701 Spruce St) offers outdoor seating and accepts credit cards. Green Engine Coffee (16 Haverford Station Rd) has outdoor seating and accepts credit cards. Little Spoon Cafe (1500 South St) also has outdoor seating and accepts credit cards. Maggie's Waterfront Cafe (9242 N Delaware Ave) offers outdoor seating and accepts credit cards.",
  'reasoning': 'These chunks are Philadelphia restaurants whose attributes list OutdoorSeating = True and BusinessAcceptsCreditCards = True (or bike_parking/accepts_credit_cards flags).'},
 {'question': 'Find family-friendly places with good for kids in Tampa.',
  'chunk_ids': ['ZXt0zo8liURaAEKD47zDRQ',
   'c7ZxVUJVjEOe9hLhvKGnDA',
   'GVNehsYuSB-fFX0FQAuI-Q'],
  'answer_example': "Ma

In [68]:
len(json_output)

117

In [45]:
points = qdrant.scroll(
    collection_name=COLLECTION,
    scroll_filter=Filter(
        must=[
            FieldCondition(
                key="__object_id__",
                match=MatchValue(value="DhZVpgeBKxoLd91MiwTxpg")
            )
        ]
    ),
    limit=100,
    with_payload=True,
    with_vectors=False
)[0]

In [46]:
points[0].payload

{'__schema_field__Business_name': 'La Fontana Della Citta',
 '__schema_field__Business_address': '1701 Spruce St',
 '__schema_field__Business_city': 'Philadelphia',
 '__schema_field__Business_state': 'PA',
 '__schema_field__Business_postal_code': '19103',
 '__schema_field__Business_latitude': 39.9477642,
 '__schema_field__Business_longitude': -75.1696977,
 '__schema_field__Business_stars': 3.0,
 '__schema_field__Business_review_count': 379,
 '__schema_field__Business_is_open': True,
 '__schema_field__Business_is_open_i': 1,
 '__schema_field__Business_attributes': '{"Alcohol": "u\'none\'", "Ambience": "{\'touristy\': False, \'hipster\': False, \'romantic\': False, \'divey\': False, \'intimate\': False, \'trendy\': False, \'upscale\': False, \'classy\': True, \'casual\': True}", "BYOBCorkage": "\'yes_free\'", "BikeParking": "True", "BusinessAcceptsCreditCards": "True", "BusinessParking": "{\'garage\': False, \'street\': True, \'validated\': False, \'lot\': False, \'valet\': False}", "Cat

In [56]:
def get_restaurant_name(business_id: str) -> str:  
    
    points = qdrant.scroll(
        collection_name=COLLECTION,
        scroll_filter=Filter(
            must=[
                FieldCondition(
                    key="__object_id__",
                    match=MatchValue(value=business_id)
                )
            ]
        ),
        limit=100,
        with_payload=True,
        with_vectors=False
    )[0]

    return points[0].payload["__schema_field__Business_name"]

In [65]:
import re

_UUID = re.compile(
    r"^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$",
    re.IGNORECASE,
)

def get_restaurant_name(business_id: str) -> str:
    if not business_id:
        return ""

    # 1) Yelp / schema id in payload (Superlinked)
    points, _ = qdrant.scroll(
        collection_name=COLLECTION,
        scroll_filter=Filter(
            must=[
                FieldCondition(
                    key="__object_id__",
                    match=MatchValue(value=business_id),
                )
            ]
        ),
        limit=1,
        with_payload=True,
        with_vectors=False,
    )

    # 2) Same but ids stored as "Business:<yelp_id>"
    if not points and business_id.startswith("Business:"):
        yelp_id = business_id.split(":", 1)[1]
        points, _ = qdrant.scroll(
            collection_name=COLLECTION,
            scroll_filter=Filter(
                must=[
                    FieldCondition(
                        key="__object_id__",
                        match=MatchValue(value=yelp_id),
                    )
                ]
            ),
            limit=1,
            with_payload=True,
            with_vectors=False,
        )

    if points and points[0].payload:
        return points[0].payload.get("__schema_field__Business_name") or business_id

    # 3) Qdrant point UUID — use retrieve, not scroll + HasIdCondition
    if _UUID.match(str(business_id).strip()):
        try:
            found = qdrant.retrieve(
                collection_name=COLLECTION,
                ids=[str(business_id).strip()],
                with_payload=True,
                with_vectors=False,
            )
        except Exception:
            return business_id
        if found and found[0].payload:
            return found[0].payload.get("__schema_field__Business_name") or business_id

    return business_id

In [66]:
get_restaurant_name("DhZVpgeBKxoLd91MiwTxpg")

'La Fontana Della Citta'

### Create Eval dataset in Langsmith

In [69]:
client = Client(api_key=os.environ["LANGSMITH_API_KEY"])

In [70]:
dataset_name = "yelp-rag-evaluation-dataset"
dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Dataset for evaluating RAG pipeline"
)

In [71]:
for item in json_output:
    client.create_example(
        dataset_id=dataset.id,
        inputs={"question": item["question"]},
        outputs={
            "ground_truth": item["answer_example"],
            "reference_context_ids": item["chunk_ids"],
            "reference_descriptions": [get_restaurant_name(id) for id in item["chunk_ids"]]
        }
    )